# Bonus task

In [24]:
import nltk
nltk.download('brown')
from nltk.corpus import brown
from collections import Counter
import re
import math

[nltk_data] Downloading package brown to /Users/fiebergli/nltk_data...
[nltk_data]   Package brown is already up-to-date!


In [25]:
def normalise_words(tokens):
    return [t.lower() for t in tokens if re.fullmatch(r"[A-Za-z]+(?:'[A-Za-z]+)?", t)]

words = normalise_words(brown.words())

In [26]:
def get_counts(words):
    word_counts = Counter(words)
    bigram_count = Counter(zip(words[:-1], words[1:]))
    return word_counts, bigram_count

def compute_pmi(words, min_freq=10):
    word_count, bigram_count = get_counts(words)

    valid_freq_words = {w for w, c in word_count.items() if c >= min_freq}
    
    filtered_bigrams = {(w1, w2): c for (w1, w2), c in bigram_count.items() if w1 in valid_freq_words and w2 in valid_freq_words}

    N_words = len(words)
    N_bigrams = len(words)-1
    scores = []

    for (w1, w2), c12 in filtered_bigrams.items():
        p_w1 = word_count[w1] / N_words
        p_w2 = word_count[w2] / N_words
        p_w1w2 = c12 / N_bigrams

        pmi = math.log2(p_w1w2 / (p_w1*p_w2))
        scores.append(((w1, w2), pmi, c12))

    return sorted(scores, key=lambda x: x[1])


In [27]:
pmi_scores = compute_pmi(words, min_freq=10)

print("Lowest 20 PMI pairs:")
for pair, score, count in pmi_scores[:20]:
    print(pair, score, count)

print("\nHighest 20 PMI pairs:")
for pair, score, count in reversed(pmi_scores[-20:]):
    print(pair, score, count)

Lowest 20 PMI pairs:
('the', 'not') -8.345869136255875 1
('the', 'but') -8.272362600262126 1
('with', 'of') -8.064480161546472 1
('an', 'the') -8.044140655714173 1
('his', 'of') -8.005289550382798 1
('the', 'his') -7.947632674860827 2
('the', 'you') -7.857434865889249 1
('a', 'he') -7.8033576879376 1
('the', 'it') -7.687070754706554 3
('the', 'would') -7.581523106358029 1
('i', 'the') -7.509591386246365 2
('he', 'of') -7.453957827542922 2
('a', 'his') -7.3546894107774765 1
('his', 'a') -7.3546894107774765 1
('the', 'from') -7.268735665327135 2
('the', 'no') -7.238038868104219 1
('of', 'of') -7.215176340361353 9
('the', 'out') -7.209429247035767 1
('the', 'the') -7.140504573814757 35
('the', 'a') -7.091881825955217 12

Highest 20 PMI pairs:
('hong', 'kong') 16.459719770244078 11
('viet', 'nam') 15.919151388881374 13
('simms', 'purdew') 15.831688547631035 12
('pathet', 'lao') 15.831688547631035 10
('herald', 'tribune') 15.56663497416059 7
('lo', 'shu') 15.459719770244076 21
('el', 'paso'

In [ ]:
#extended version for step 3
def compute_pmi(words, min_freq=10, positive=True):
    word_count, bigram_count = get_counts(words)

    valid_freq_words = {w for w, c in word_count.items() if c >= min_freq}
    
    filtered_bigrams = {(w1, w2): c for (w1, w2), c in bigram_count.items() if w1 in valid_freq_words and w2 in valid_freq_words}

    N_words = len(words)
    N_bigrams = len(words)-1
    scores = []

    for (w1, w2), c12 in filtered_bigrams.items():
        p_w1 = word_count[w1] / N_words
        p_w2 = word_count[w2] / N_words
        p_w1w2 = c12 / N_bigrams

        pmi = math.log2(p_w1w2 / (p_w1*p_w2))
        if positive:
            pmi = max(0, pmi)

        scores.append(((w1, w2), pmi, c12))

    return sorted(scores, key=lambda x: x[1])